In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import yfinance as yf

from analytics.regime_classifier import RegimeClassifier, MarketRegime
from analytics.temporal import detect_regime_change
from visualization.time_series import plot_regime_timeline

In [ ]:
# Load historical VIX data as proxy for ATM vol
vix = yf.download('^VIX', start='2020-01-01', end='2024-12-31')
spy = yf.download('SPY', start='2020-01-01', end='2024-12-31')

print(f"Loaded {len(vix)} days of VIX data")
print(f"Loaded {len(spy)} days of SPY data")

In [ ]:
# Create synthetic surface metrics based on VIX
# Note: In production, you'd use actual historical option data

vix_level = vix['Close'] / 100  # Convert to decimal

# Synthetic skew - tends to increase with VIX
skew = -0.05 - 0.1 * (vix_level - 0.2)

# Synthetic curvature - convex relationship with VIX
curvature = 0.01 + 0.05 * (vix_level - 0.15) ** 2

# Synthetic butterfly
butterfly = curvature * 0.5

metrics_df = pd.DataFrame({
    'atm_vol': vix_level,
    'skew': skew,
    'curvature': curvature,
    'butterfly': butterfly
}, index=vix.index)

metrics_df.dropna(inplace=True)
metrics_df.head()

In [ ]:
# Initialize classifier with historical data for percentile calculation
classifier = RegimeClassifier(
    history_window=252,
    pre_stress_threshold=70,
    elevated_threshold=85,
    acute_threshold=95
)

# Build historical reference
for i in range(min(252, len(metrics_df))):
    row = metrics_df.iloc[i]
    classifier.update_history(
        atm_vol=row['atm_vol'],
        skew=row['skew'],
        curvature=row['curvature'],
        butterfly=row['butterfly']
    )

In [ ]:
# Classify each day
regimes = []

for i in range(252, len(metrics_df)):
    row = metrics_df.iloc[i]
    result = classifier.classify(
        atm_vol=row['atm_vol'],
        skew=row['skew'],
        curvature=row['curvature'],
        butterfly=row['butterfly']
    )
    regimes.append({
        'date': metrics_df.index[i],
        'regime': result['current_regime'],
        'confidence': result['confidence'],
        'atm_vol': row['atm_vol']
    })
    
    # Update history
    classifier.update_history(
        atm_vol=row['atm_vol'],
        skew=row['skew'],
        curvature=row['curvature'],
        butterfly=row['butterfly']
    )

regimes_df = pd.DataFrame(regimes)
regimes_df.set_index('date', inplace=True)

print(f"Classified {len(regimes_df)} days")
regimes_df['regime'].value_counts()

In [ ]:
# Visualize regimes over time with SPY returns
spy_aligned = spy['Close'].reindex(regimes_df.index)
spy_returns = spy_aligned.pct_change()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=['SPY Price', 'VIX (ATM Vol)', 'Classified Regime'],
    row_heights=[0.35, 0.35, 0.3]
)

# SPY price
fig.add_trace(
    go.Scatter(x=regimes_df.index, y=spy_aligned, name='SPY', line=dict(color='blue')),
    row=1, col=1
)

# VIX
fig.add_trace(
    go.Scatter(x=regimes_df.index, y=regimes_df['atm_vol']*100, name='VIX', 
               line=dict(color='orange')),
    row=2, col=1
)

# Regime map
regime_map = {'calm': 1, 'pre_stress': 2, 'recovery': 2.5, 'elevated': 3, 'acute': 4}
regime_colors = {'calm': 'green', 'pre_stress': 'yellow', 'recovery': 'blue', 
                 'elevated': 'orange', 'acute': 'red'}

for regime in regime_map.keys():
    mask = regimes_df['regime'] == regime
    if mask.any():
        fig.add_trace(
            go.Scatter(
                x=regimes_df.index[mask],
                y=[regime_map[regime]] * mask.sum(),
                mode='markers',
                marker=dict(color=regime_colors[regime], size=5),
                name=regime.title()
            ),
            row=3, col=1
        )

fig.update_layout(height=800, title='Backtest: Regime Classification 2020-2024')
fig.show()

In [ ]:
# Analyze forward returns by regime
forward_returns = []

for horizon in [1, 5, 10, 21]:
    fwd_ret = spy_returns.shift(-horizon).rolling(horizon).sum()
    
    for regime in ['calm', 'pre_stress', 'elevated', 'acute']:
        mask = regimes_df['regime'] == regime
        ret = fwd_ret[mask].mean()
        vol = fwd_ret[mask].std()
        
        forward_returns.append({
            'horizon': horizon,
            'regime': regime,
            'mean_return': ret,
            'volatility': vol,
            'sharpe': ret / vol if vol > 0 else 0
        })

fwd_df = pd.DataFrame(forward_returns)
fwd_df.pivot(index='regime', columns='horizon', values='mean_return').style.format('{:.2%}')

In [ ]:
# Evaluate pre-stress signal effectiveness
# Check if pre-stress correctly preceded elevated/acute

pre_stress_dates = regimes_df[regimes_df['regime'] == 'pre_stress'].index
stress_dates = regimes_df[regimes_df['regime'].isin(['elevated', 'acute'])].index

correct_signals = 0
false_signals = 0

for date in pre_stress_dates:
    # Look forward 5 days
    future_dates = regimes_df.loc[date:].head(6).index[1:]
    future_regimes = regimes_df.loc[future_dates, 'regime']
    
    if any(future_regimes.isin(['elevated', 'acute'])):
        correct_signals += 1
    else:
        false_signals += 1

print(f"Pre-Stress Signal Effectiveness:")
print(f"  Total pre-stress signals: {len(pre_stress_dates)}")
print(f"  Correctly predicted stress: {correct_signals}")
print(f"  False alarms: {false_signals}")
print(f"  Precision: {correct_signals / (correct_signals + false_signals):.1%}")